<h1 style="color:DodgerBlue">Спринт ревью #3 ООП</h1>

**Тема:** «Полиморфизм и Generic классы»

**Выполнила:** Решетова Арина, группа СМАРТБ-25-1

----

### Описание задания

Программа моделирует **компьютер** и его **компоненты**. Что используется:

- абстрактный класс `Component` и наследники `CPU`, `RAM`, `GPU`, `HDD` (наследование);
- интерфейс `IInstallable` с методом `Install()`;
- виртуальный метод `GetInfo()`, который перекрывается в каждом наследнике (**полиморфизм, override**);
- перегрузка `ToString()` и операторов `==` / `!=`;
- generic-методы `AddComponent<T>` и `RemoveComponent<T>` с ограничениями `where`;
- статический класс `ComputerFactory` с generic-методом `Build<T>`.

Ячейки ниже нужно запускать **по порядку** сверху вниз.

#### 1. Интерфейс `IInstallable` и базовый класс `Component`

In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

// Интерфейс: любой устанавливаемый компонент обязан уметь устанавливаться
public interface IInstallable
{
    void Install();
}

// Базовый абстрактный класс для всех компонентов компьютера
public abstract class Component
{
    private decimal _price;

    public string Name { get; set; }
    public string Manufacturer { get; set; }

    // В сеттере цены проверяем, что она не отрицательная
    public decimal Price
    {
        get => _price;
        set
        {
            if (value < 0) throw new ArgumentException("Цена не может быть отрицательной");
            _price = value;
        }
    }

    protected Component(string name, string manufacturer, decimal price)
    {
        Name = name;
        Manufacturer = manufacturer;
        Price = price;
    }

    // Виртуальный метод: потомки перекроют его (override)
    public virtual void GetInfo()
    {
        Console.WriteLine($"Компонент: {Name} ({Manufacturer}), цена: {Price} руб.");
    }

    // Переопределяем ToString() для удобного вывода
    public override string ToString()
    {
        return $"{GetType().Name}: {Name} ({Manufacturer}) - {Price} руб.";
    }
}

#### 2. Конкретные компоненты: `CPU`, `RAM`, `GPU`, `HDD`

In [ ]:
// ---------- CPU ----------
public class CPU : Component, IInstallable
{
    public int Cores { get; set; }
    public double FrequencyGHz { get; set; }

    public CPU(string name, string manufacturer, decimal price, int cores, double frequencyGHz)
        : base(name, manufacturer, price)
    {
        Cores = cores;
        FrequencyGHz = frequencyGHz;
    }

    public override void GetInfo()
    {
        Console.WriteLine($"[CPU] {Name} ({Manufacturer}): {Cores} ядер, {FrequencyGHz} ГГц, {Price} руб.");
    }

    public void Install()
    {
        Console.WriteLine($"Процессор {Name} установлен в сокет материнской платы.");
    }

    // Перегрузка операторов сравнения
    public static bool operator ==(CPU a, CPU b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Manufacturer == b.Manufacturer
            && a.Cores == b.Cores && a.FrequencyGHz == b.FrequencyGHz;
    }
    public static bool operator !=(CPU a, CPU b) => !(a == b);

    // Вместе с == принято переопределять Equals и GetHashCode
    public override bool Equals(object obj) => obj is CPU other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, Manufacturer, Cores, FrequencyGHz);
}

// ---------- RAM ----------
public class RAM : Component, IInstallable
{
    public int CapacityGB { get; set; }
    public string MemoryType { get; set; }   // например, DDR4 / DDR5
    public int SpeedMHz { get; set; }

    public RAM(string name, string manufacturer, decimal price, int capacityGB, string memoryType, int speedMHz)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
        MemoryType = memoryType;
        SpeedMHz = speedMHz;
    }

    public override void GetInfo()
    {
        Console.WriteLine($"[RAM] {Name} ({Manufacturer}): {CapacityGB} ГБ {MemoryType}, {SpeedMHz} МГц, {Price} руб.");
    }

    public void Install()
    {
        Console.WriteLine($"Модуль памяти {Name} вставлен в слот DIMM.");
    }

    public static bool operator ==(RAM a, RAM b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Manufacturer == b.Manufacturer
            && a.CapacityGB == b.CapacityGB && a.MemoryType == b.MemoryType && a.SpeedMHz == b.SpeedMHz;
    }
    public static bool operator !=(RAM a, RAM b) => !(a == b);

    public override bool Equals(object obj) => obj is RAM other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, Manufacturer, CapacityGB, MemoryType, SpeedMHz);
}

// ---------- GPU ----------
public class GPU : Component, IInstallable
{
    public int VideoMemoryGB { get; set; }
    public int CoreClockMHz { get; set; }

    public GPU(string name, string manufacturer, decimal price, int videoMemoryGB, int coreClockMHz)
        : base(name, manufacturer, price)
    {
        VideoMemoryGB = videoMemoryGB;
        CoreClockMHz = coreClockMHz;
    }

    public override void GetInfo()
    {
        Console.WriteLine($"[GPU] {Name} ({Manufacturer}): {VideoMemoryGB} ГБ видеопамяти, {CoreClockMHz} МГц, {Price} руб.");
    }

    public void Install()
    {
        Console.WriteLine($"Видеокарта {Name} установлена в слот PCI-E.");
    }

    public static bool operator ==(GPU a, GPU b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Manufacturer == b.Manufacturer
            && a.VideoMemoryGB == b.VideoMemoryGB && a.CoreClockMHz == b.CoreClockMHz;
    }
    public static bool operator !=(GPU a, GPU b) => !(a == b);

    public override bool Equals(object obj) => obj is GPU other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, Manufacturer, VideoMemoryGB, CoreClockMHz);
}

// ---------- HDD ----------
public class HDD : Component, IInstallable
{
    public int CapacityGB { get; set; }
    public int RPM { get; set; }   // скорость вращения шпинделя

    public HDD(string name, string manufacturer, decimal price, int capacityGB, int rpm)
        : base(name, manufacturer, price)
    {
        CapacityGB = capacityGB;
        RPM = rpm;
    }

    public override void GetInfo()
    {
        Console.WriteLine($"[HDD] {Name} ({Manufacturer}): {CapacityGB} ГБ, {RPM} об/мин, {Price} руб.");
    }

    public void Install()
    {
        Console.WriteLine($"Жёсткий диск {Name} подключён к SATA.");
    }

    public static bool operator ==(HDD a, HDD b)
    {
        if (ReferenceEquals(a, b)) return true;
        if (a is null || b is null) return false;
        return a.Name == b.Name && a.Manufacturer == b.Manufacturer
            && a.CapacityGB == b.CapacityGB && a.RPM == b.RPM;
    }
    public static bool operator !=(HDD a, HDD b) => !(a == b);

    public override bool Equals(object obj) => obj is HDD other && this == other;
    public override int GetHashCode() => HashCode.Combine(Name, Manufacturer, CapacityGB, RPM);
}

#### 3. Классы `Computer` и `ComputerFactory`

In [ ]:
// ---------- Computer ----------
public class Computer
{
    private readonly List<Component> _components = new List<Component>();

    public string Name { get; set; }

    public Computer(string name)
    {
        Name = name;
    }

    public IReadOnlyList<Component> Components => _components;

    // Generic-метод: принимает только то, что одновременно Component и IInstallable
    public void AddComponent<T>(T component) where T : Component, IInstallable
    {
        component.Install();          // вызываем установку через интерфейс
        _components.Add(component);
        Console.WriteLine($"  -> {component.Name} добавлен в {Name}");
    }

    // Generic-метод удаления
    public bool RemoveComponent<T>(T component) where T : Component
    {
        bool removed = _components.Remove(component);   // сравнение идёт через Equals
        Console.WriteLine(removed
            ? $"  -> {component.Name} удалён из {Name}"
            : $"  -> {component.Name} не найден в {Name}");
        return removed;
    }

    public decimal GetTotalPrice()
    {
        return _components.Sum(c => c.Price);
    }

    public void ShowConfiguration()
    {
        Console.WriteLine($"=== Конфигурация «{Name}» ===");
        foreach (var c in _components)
            c.GetInfo();               // полиморфизм: у каждого типа своя версия GetInfo()
        Console.WriteLine($"Итого: {GetTotalPrice()} руб.");
    }
}

// ---------- Статическая фабрика ----------
public static class ComputerFactory
{
    // Generic-метод: собирает компьютер из любого набора однотипных устанавливаемых компонентов
    public static Computer Build<T>(string name, params T[] parts) where T : Component, IInstallable
    {
        var pc = new Computer(name);
        foreach (var part in parts)
            pc.AddComponent(part);
        return pc;
    }

    public static Computer CreateOfficeComputer()
    {
        var pc = new Computer("Офисный ПК");
        pc.AddComponent(new CPU("Core i3-12100", "Intel", 9500m, 4, 3.3));
        pc.AddComponent(new RAM("Vengeance 8GB", "Corsair", 2500m, 8, "DDR4", 3200));
        pc.AddComponent(new HDD("Barracuda 1TB", "Seagate", 3500m, 1000, 7200));
        return pc;
    }

    public static Computer CreateGamingComputer()
    {
        var pc = new Computer("Игровой ПК");
        pc.AddComponent(new CPU("Ryzen 7 7800X3D", "AMD", 35000m, 8, 4.2));
        pc.AddComponent(new RAM("Fury 32GB", "Kingston", 9000m, 32, "DDR5", 6000));
        pc.AddComponent(new GPU("RTX 4070", "NVIDIA", 55000m, 12, 1920));
        pc.AddComponent(new HDD("Barracuda 2TB", "Seagate", 5000m, 2000, 7200));
        return pc;
    }

    public static Computer CreateServer()
    {
        var pc = new Computer("Сервер");
        pc.AddComponent(new CPU("Xeon Silver 4310", "Intel", 60000m, 12, 2.1));
        pc.AddComponent(new RAM("ECC 64GB", "Samsung", 20000m, 64, "DDR4", 3200));
        pc.AddComponent(new HDD("Ultrastar 4TB", "WD", 12000m, 4000, 7200));
        return pc;
    }
}

#### 4. Демонстрационное приложение

In [ ]:
// ========== Демонстрационное приложение ==========
Console.WriteLine("--- 1. Создаём компоненты ---");
var cpu1 = new CPU("Core i5-12400", "Intel", 15000m, 6, 2.5);
var cpu2 = new CPU("Core i5-12400", "Intel", 15000m, 6, 2.5);   // такой же по характеристикам
var cpu3 = new CPU("Ryzen 5 5600", "AMD", 13000m, 6, 3.5);
var ram  = new RAM("Fury 16GB", "Kingston", 4500m, 16, "DDR4", 3200);
var gpu  = new GPU("RTX 3060", "NVIDIA", 30000m, 12, 1777);
var hdd  = new HDD("Blue 1TB", "WD", 3200m, 1000, 7200);

Console.WriteLine("\n--- 2. ToString() ---");
Console.WriteLine(cpu1);
Console.WriteLine(ram);

Console.WriteLine("\n--- 3. Полиморфизм GetInfo() через базовый тип ---");
Component[] all = { cpu1, ram, gpu, hdd };
foreach (var c in all) c.GetInfo();

Console.WriteLine("\n--- 4. Перегруженные операторы == и != ---");
Console.WriteLine($"cpu1 == cpu2 : {cpu1 == cpu2}");   // True
Console.WriteLine($"cpu1 == cpu3 : {cpu1 == cpu3}");   // False
Console.WriteLine($"cpu1 != cpu3 : {cpu1 != cpu3}");   // True

Console.WriteLine("\n--- 5. Computer: AddComponent<T> / GetTotalPrice ---");
var myPc = new Computer("Мой ПК");
myPc.AddComponent(cpu1);
myPc.AddComponent(ram);
myPc.AddComponent(gpu);
myPc.AddComponent(hdd);
myPc.ShowConfiguration();

Console.WriteLine("\n--- 6. RemoveComponent<T> ---");
myPc.RemoveComponent(gpu);
myPc.RemoveComponent(cpu2);   // удалится cpu1, т.к. cpu2 равен ему по Equals
myPc.RemoveComponent(cpu3);   // такого в ПК нет
Console.WriteLine($"Стоимость после удаления: {myPc.GetTotalPrice()} руб.");

Console.WriteLine("\n--- 7. ComputerFactory ---");
ComputerFactory.CreateOfficeComputer().ShowConfiguration();
Console.WriteLine();
ComputerFactory.CreateGamingComputer().ShowConfiguration();
Console.WriteLine();
ComputerFactory.CreateServer().ShowConfiguration();

Console.WriteLine("\n--- 8. Generic-метод Build<T> ---");
var ramPc = ComputerFactory.Build("ПК из двух модулей RAM",
    new RAM("Module A", "Kingston", 2000m, 8, "DDR4", 3200),
    new RAM("Module B", "Kingston", 2000m, 8, "DDR4", 3200));
ramPc.ShowConfiguration();